# Inversion Example: FLEXPART Inverse Modeling Training

We need to create some true emissions and perturb them to create a priori emissions. In real practice, you would have observations and a priori emissions from other sources, but for this example, we will create them artificially.

You can choose the perturbations for the emissions. For example, you can use `[0.5, -0.5]` to add 50% of the first emission value and subtract 50% of the second emission value. You can also add more perturbations if needed, such as `[0.5, -0.5, 0.2]` for three emission sources.

Here you follow your results from the previous tasks in the [Emissions.ipynb](Emissions.ipynb) notebook. You can use the same true emissions and define perturbations here.

In [ ]:
import functions as fs
############################################## SETTINGS ################################################################
grid_time_file = "???" # path to the grid time file
colorbar_limits = [0.1, 1000] # sm³/kg
height = 100  #[m] height of the level to plot
map_coordinates = [23, 35, 34, 45] # [lon_min, lon_max, lat_min, lat_max]
lat, lon, time, release_times, height, conc, f = fs.read_grid_time_file(grid_time_file)

In [ ]:
########################################## specify emission sources ####################################################
true_emissions = []
# e.g. use some locations like this:
true_emissions = fs.add_source(true_emissions, lat=38.5, lon=29.5, val=1) #  Emissions in [ng/m²s]
true_emissions = fs.add_source(true_emissions, lat=40.5, lon=27.5, val=5)
# true_emissions = fs.add_source(true_emissions, lat=43.5, lon=33.5, val=3)

# add 50% of first emission value, substract 50% of second emission value
# add more if needed, e.g. [0.5, -0.5, 0.2] for three emission sources
perturbations = [0.5, -0.5]

#########################################  plot true/a priori emissions   #######################################################

a_priori_emissions = fs.perturb_emissions(perturbations, true_emissions)
fig = fs.concentration_and_emissions_before_inversion(lat, lon, height, release_times , conc, true_emissions, a_priori_emissions)

You should see a misfit of the true vs the perturb (a priori) emissions. The goal of the inversion is to reduce this misfit and estimate the true emissions based on the emission sensitivity.

# Mini inversion

We can now perform a mini inversion using the FLEXPART inverse modeling framework. The goal is to estimate the emissions from the observed concentrations at specific locations and times.

Question

- How do the estimated emissions compare to the a priori / true emissions?
- Does the inversion work?
- What effects do the observation and emission errors have on the inversion results?

In [ ]:
#########################################  perform inversion   #######################################################
# represent the uncertainty in the observations / measurement uncertainty.
observation_error = 3   # [ng/m²s]
# represent the uncertainty in the emission sensitivity / model uncertainty.
emission_error = 0.2 # fraction [1] of the a priori emissions

# calculate the timeseries
timeseries = fs.calculate_timeseries(lat, lon, height, conc,  true_emissions)
H, Ht = fs.calculate_Transport_matrix_H(lat, lon, conc, height, true_emissions)
optimized_emissions = fs.inversion(a_priori_emissions, emission_error, observation_error, timeseries, H, Ht)
fig = fs.concentration_and_emissions_after_inversion(lat, lon, height, release_times, conc, true_emissions, a_priori_emissions, optimized_emissions)